# FZ Design of Experiments with `fzd`

`fzd` drives **iterative, adaptive sampling** via pluggable algorithms.
Each iteration asks the algorithm for new design points, runs the model,
and feeds results back until the algorithm converges.

| # | Problem | Algorithm | Purpose |
|---|---------|-----------|---------|
| A | Exploration of Himmelblau's surface | Random Sampling | Visualise the landscape |
| B | 1-D root-finding on a noisy curve | Brent | Locate zero crossing |
| C | 2-D optimisation of Rosenbrock function | BFGS | Find global minimum |
| D | Monte Carlo π estimation | Monte Carlo | Estimate an integral |


In [1]:
import json, shutil
from pathlib import Path
import fz
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fz.set_log_level("WARNING")

# Always start fresh so fzd cache never picks up stale results
WORK = Path("work_04_doe")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir()
(WORK / "models").mkdir()

# Resolve algorithm paths
ALGO_BASE = Path("../algorithms")
def algo(name):
    candidates = [
        ALGO_BASE / f"{name}.py",
        Path(f"/home/richet/Sync/Open/Funz/github/fz/examples/algorithms/{name}.py"),
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"Algorithm {name} not found")


### Setup: simulators

In [2]:
# ---- Himmelblau: f(x,y) = (x^2+y-11)^2 + (x+y^2-7)^2  (4 minima at 0) ----
SIM_H = WORK / "models" / "himmelblau.py"
SIM_H.write_text('''#!/usr/bin/env python3
params = {}
with open("h.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
x, y = params["x"], params["y"]
f = (x**2 + y - 11)**2 + (x + y**2 - 7)**2
with open("output.txt", "w") as fh:
    fh.write(f"f = {f:.10f}\\n")
''')
TMPL_H = WORK / "h.in"
TMPL_H.write_text("x = ${x~0.0}\ny = ${y~0.0}\n")
MODEL_H = {"varprefix": "$", "delim": "{}", "commentline": "#",
           "output": {"f": "grep '^f = ' output.txt | awk '{print $3}'"}}
CALC_H = f"sh://python3 {SIM_H.resolve()}"

# ---- Rosenbrock: f(x,y) = (1-x)^2 + 100*(y-x^2)^2  (min at (1,1)=0) ----
SIM_R = WORK / "models" / "rosenbrock.py"
SIM_R.write_text('''#!/usr/bin/env python3
params = {}
with open("r.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
x, y = params["x"], params["y"]
f = (1 - x)**2 + 100 * (y - x**2)**2
with open("output.txt", "w") as fh:
    fh.write(f"f = {f:.10f}\\n")
''')
TMPL_R = WORK / "r.in"
TMPL_R.write_text("x = ${x~0.0}\ny = ${y~0.0}\n")
MODEL_R = {"varprefix": "$", "delim": "{}", "commentline": "#",
           "output": {"f": "grep '^f = ' output.txt | awk '{print $3}'"}}
CALC_R = f"sh://python3 {SIM_R.resolve()}"

# ---- 1-D noisy curve: g(x) = x^3 - 4x + cos(x)  (root near x≈1.86) ----
SIM_1D = WORK / "models" / "curve1d.py"
SIM_1D.write_text('''#!/usr/bin/env python3
import math
params = {}
with open("c.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
x = params["x"]
g = x**3 - 4*x + math.cos(x)
with open("output.txt", "w") as fh:
    fh.write(f"g = {g:.10f}\\n")
''')
TMPL_1D = WORK / "c.in"
TMPL_1D.write_text("x = ${x~0.0}\n")
MODEL_1D = {"varprefix": "$", "delim": "{}", "commentline": "#",
            "output": {"g": "grep '^g = ' output.txt | awk '{print $3}'"}}
CALC_1D = f"sh://python3 {SIM_1D.resolve()}"

print("All simulators ready.")


All simulators ready.


### Setup: Monte Carlo π simulator

In [3]:
SIM_MC = WORK / "models" / "montecarlo_pi.py"
SIM_MC.write_text('''#!/usr/bin/env python3
import random
params = {}
with open("mc.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
n = int(params["n_samples"])
seed = int(params["seed"])
random.seed(seed)
inside = sum(1 for _ in range(n)
             if random.random()**2 + random.random()**2 <= 1.0)
pi_est = 4 * inside / n
error  = abs(pi_est - 3.14159265358979)
with open("output.txt", "w") as fh:
    fh.write(f"pi_estimate = {pi_est:.8f}\\n")
    fh.write(f"error = {error:.8f}\\n")
''')
TMPL_MC = WORK / "mc.in"
TMPL_MC.write_text("n_samples = ${n_samples~1000}\nseed = ${seed~42}\n")
MODEL_MC = {
    "varprefix": "$", "delim": "{}", "commentline": "#",
    "output": {
        "pi_estimate": "grep '^pi_estimate' output.txt | awk '{print $3}'",
        "error":       "grep '^error '     output.txt | awk '{print $3}'",
    },
}
CALC_MC = f"sh://python3 {SIM_MC.resolve()}"
print("Monte Carlo simulator ready.")


Monte Carlo simulator ready.


## A · Random sampling — explore Himmelblau's surface

In [4]:
result_rs = fz.fzd(
    str(TMPL_H),
    {"x": "[-5.0;5.0]", "y": "[-5.0;5.0]"},
    MODEL_H,
    output_expression="f",
    algorithm=algo("randomsampling"),
    algorithm_options={"nvalues": 40, "seed": 0},
    calculators=[CALC_H],
    analysis_dir=str(WORK / "doe_random"),
)
df_rs = result_rs["XY"]
df_rs["f"] = df_rs["f"].astype(float)
print(f"Sampled {len(df_rs)} points.  f range: [{df_rs['f'].min():.2f}, {df_rs['f'].max():.2f}]")
print("Summary:", result_rs["summary"])


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◢ [█>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   2% (1/40) ETA: 21s

...

◥ [███████████████████████████████████████>]  97% (39/40) ETA: 0s

◢ [███████████████████████████████████████>]  97% (39/40) ETA: 0s

◣ [███████████████████████████████████████>]  97% (39/40) ETA: 0s

  [████████████████████████████████████████] 100% (40/40) Total: 20s

Sampled 40 points.  f range: [2.38, 536.56]
Summary: /home/richet/Sync/Open/Funz/github/fz/examples/algorithms/randomsampling.py completed: 1 iterations, 40 evaluations (40 valid)


In [5]:
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
ff = (xx**2 + yy - 11)**2 + (xx + yy**2 - 7)**2

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(xx, yy, ff, levels=40, cmap="viridis")
ax.scatter(df_rs["x"].astype(float), df_rs["y"].astype(float),
           c="white", s=30, edgecolors="black", lw=0.5,
           label=f"samples (n={len(df_rs)})")
plt.colorbar(c, label="f(x,y)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Himmelblau's function — random sampling exploration")
ax.legend(); plt.tight_layout()
plt.savefig(str(WORK / "random_sampling.png"), dpi=100)
plt.show()
print("Plot saved.")


Plot saved.


/tmp/ipykernel_51683/4078916932.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## B · 1-D minimisation with Brent's method

The bundled `brent.py` is a **1-D optimiser** (minimiser) based on parabolic interpolation + golden section.  We minimise a smooth unimodal function to find its minimum.

In [6]:
# ---- 1-D smooth function with a clear minimum inside [0, 3] ----
# f(x) = (x - 1.6)^2 + 0.2*sin(5*x)   → minimum near x ≈ 1.6
SIM_1D.write_text('''#!/usr/bin/env python3
import math
params = {}
with open("c.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())
x = params["x"]
g = (x - 1.6)**2 + 0.2 * math.sin(5 * x)
with open("output.txt", "w") as fh:
    fh.write(f"g = {g:.10f}\\n")
''')

result_brent = fz.fzd(
    str(TMPL_1D),
    {"x": "[0.0;3.0]"},
    MODEL_1D,
    output_expression="g",        # minimise this
    algorithm=algo("brent"),
    algorithm_options={"tol": 1e-6, "max_iter": 50},
    calculators=[CALC_1D],
    analysis_dir=str(WORK / "doe_brent"),
)
df_brent = result_brent["XY"]
print(result_brent["summary"])
best_b = df_brent.loc[df_brent["g"].astype(float).idxmin()]
print(f"Minimum found:  x = {float(best_b['x']):.6f},  f = {float(best_b['g']):.8f}")
print(f"Analytical min: x ≈ 2.008, f ≈ 0.051  (verified with scipy)")


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/3) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/3) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/3) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/3) ETA: ...

◢ [█████████████>░░░░░░░░░░░░░░░░░░░░░░░░░░]  33% (1/3) ETA: 1s

...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

  [████████████████████████████████████████] 100% (1/1) Total: 0s

/home/richet/Sync/Open/Funz/github/fz/examples/algorithms/brent.py completed: 50 iterations, 52 evaluations (52 valid)
Minimum found:  x = 1.968750,  f = 0.05461224
Analytical min: x ≈ 2.008, f ≈ 0.051  (verified with scipy)


In [7]:
import math
xs_plot = np.linspace(0, 3, 300)
gs_plot = [(x - 1.6)**2 + 0.2 * math.sin(5 * x) for x in xs_plot]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xs_plot, gs_plot, "k-", lw=2, label="f(x) = (x−1.6)² + 0.2·sin(5x)")
ax.scatter(df_brent["x"].astype(float), df_brent["g"].astype(float),
           c=range(len(df_brent)), cmap="plasma", s=60, zorder=5,
           label=f"Brent evaluations (n={len(df_brent)})")
ax.scatter([float(best_b["x"])], [float(best_b["g"])],
           c="red", s=150, marker="*", zorder=10,
           label=f"Best: x={float(best_b['x']):.4f}")
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.set_title("Brent 1-D minimisation")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(str(WORK / "brent_result.png"), dpi=100)
plt.show()


/tmp/ipykernel_51683/1546914567.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## C · 2-D optimisation with BFGS

In [8]:
result_bfgs = fz.fzd(
    str(TMPL_R),
    {"x": "[-2.0;2.0]", "y": "[-1.0;3.0]"},
    MODEL_R,
    output_expression="f",
    algorithm=algo("bfgs"),
    algorithm_options={"max_iter": 100, "tol": 1e-6},
    calculators=[CALC_R],
    analysis_dir=str(WORK / "doe_bfgs"),
)
df_bfgs = result_bfgs["XY"]
df_bfgs["f"] = df_bfgs["f"].astype(float)
best = df_bfgs.loc[df_bfgs["f"].idxmin()]
print(result_bfgs["summary"])
print(f"Best: x={float(best['x']):.6f}, y={float(best['y']):.6f}, f={best['f']:.8f}")
print(f"Theoretical minimum: x=1, y=1, f=0")


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

  [████████████████████████████████████████] 100% (1/1) Total: 0s

...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/1) ETA: ...

  [████████████████████████████████████████] 100% (1/1) Total: 0s

/home/richet/Sync/Open/Funz/github/fz/examples/algorithms/bfgs.py completed: 100 iterations, 100 evaluations (100 valid)
Best: x=0.722547, y=0.502861, f=0.11389399
Theoretical minimum: x=1, y=1, f=0


In [9]:
xx, yy = np.meshgrid(np.linspace(-2, 2, 300), np.linspace(-1, 3, 300))
ff = (1 - xx)**2 + 100 * (yy - xx**2)**2

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(xx, yy, np.log1p(ff), levels=50, cmap="hot_r")
ax.plot(df_bfgs["x"].astype(float), df_bfgs["y"].astype(float),
        "w.-", lw=1.5, ms=6, label="BFGS path")
ax.scatter([1], [1], c="lime", s=200, marker="*", zorder=10, label="True min (1,1)")
ax.scatter([float(best["x"])], [float(best["y"])], c="cyan", s=100, marker="D",
           zorder=10, label=f"Found ({float(best['x']):.3f},{float(best['y']):.3f})")
plt.colorbar(c, label="log(1+f)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Rosenbrock function — BFGS optimisation")
ax.legend(); plt.tight_layout()
plt.savefig(str(WORK / "bfgs_result.png"), dpi=100)
plt.show()


/tmp/ipykernel_51683/1534465644.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## D · Monte Carlo π estimation

Sweep `seed` and `n_samples` to see how estimate variance scales with 1/√n.

In [10]:
df_mc = fz.fzr(
    str(TMPL_MC),
    {"n_samples": [100, 1_000, 10_000, 100_000],
     "seed": list(range(10))},
    MODEL_MC,
    results_dir=str(WORK / "results_mc"),
    calculators=[CALC_MC],
)
df_mc["n_samples"]   = df_mc["n_samples"].astype(int)
df_mc["pi_estimate"] = pd.to_numeric(df_mc["pi_estimate"], errors="coerce")
df_mc["error"]       = pd.to_numeric(df_mc["error"],       errors="coerce")

stats = df_mc.groupby("n_samples")["pi_estimate"].agg(["mean", "std", "min", "max"])
stats.columns = ["mean_π", "std_π", "min_π", "max_π"]
print("Monte Carlo π estimation statistics:")
print(stats.to_string())


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/40) ETA: ...

◢ [█>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   2% (1/40) ETA: 20s

...

◢ [███████████████████████████████████████>]  97% (39/40) ETA: 0s

◣ [███████████████████████████████████████>]  97% (39/40) ETA: 0s

◤ [███████████████████████████████████████>]  97% (39/40) ETA: 0s

  [████████████████████████████████████████] 100% (40/40) Total: 20s

Monte Carlo π estimation statistics:
             mean_π     std_π    min_π    max_π
n_samples                                      
100        3.080000  0.176887  2.84000  3.40000
1000       3.136000  0.028659  3.09200  3.19200
10000      3.139920  0.010161  3.12040  3.15320
100000     3.142448  0.005498  3.13472  3.15316


In [11]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ns = sorted(df_mc["n_samples"].unique())
positions = list(range(len(ns)))
for i, n in enumerate(ns):
    vals = df_mc[df_mc["n_samples"] == n]["pi_estimate"].dropna()
    ax1.boxplot(vals, positions=[i], widths=0.6, patch_artist=True,
                boxprops=dict(facecolor=plt.cm.Blues(0.4+0.15*i)))
ax1.axhline(np.pi, color="red", ls="--", label=f"True π = {np.pi:.5f}")
ax1.set_xticks(positions); ax1.set_xticklabels([str(n) for n in ns])
ax1.set_xlabel("n_samples"); ax1.set_ylabel("π estimate")
ax1.set_title("π estimate distribution"); ax1.legend(); ax1.grid(alpha=.3)

ax2.loglog(stats.index, stats["std_π"], "o-b", label="σ(π estimate)")
ax2.loglog(stats.index, 1/np.sqrt(stats.index), "r--", label="1/√n (theory)")
ax2.set_xlabel("n_samples"); ax2.set_ylabel("Std deviation")
ax2.set_title("Monte Carlo convergence rate"); ax2.legend(); ax2.grid(alpha=.3, which="both")

plt.tight_layout()
plt.savefig(str(WORK / "montecarlo_pi.png"), dpi=100)
plt.show()


/tmp/ipykernel_51683/2710727620.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## E · Comparing algorithms

In [12]:
print("Algorithm comparison summary:")
print(f"  Random sampling  : {len(df_rs)} evaluations, "
      f"best f={df_rs['f'].min():.4f}")
print(f"  Brent 1-D minimiser: {len(df_brent)} evaluations, "
      f"f_min={df_brent['g'].astype(float).min():.6f}")
print(f"  BFGS optimiser   : {len(df_bfgs)} evaluations, "
      f"f_min={df_bfgs['f'].min():.2e}")


Algorithm comparison summary:
  Random sampling  : 40 evaluations, best f=2.3764
  Brent 1-D minimiser: 52 evaluations, f_min=0.054612
  BFGS optimiser   : 100 evaluations, f_min=1.14e-01
